# Phase 3: Exploratory Data Analysis (EDA)

**Project:** Customer Churn Prediction & Retention Analytics  
**Cleaned Dataset:** `data/processed/telco_churn_clean.csv`  
**Target Variable:** `Churn Value` (0 = Retained, 1 = Churned)  

### Objectives:
1. Analyze overall churn distribution and baseline churn rate.
2. Calculate category-level **churn rates** ($\frac{\text{Churned}}{\text{Total}} \times 100$) across key contract, service, demographic, and billing dimensions.
3. Investigate tenure cohorts and their relationship with churn propensity.
4. Compare financial distributions (Monthly Charges, Total Charges) between churned and retained customers.
5. Evaluate correlation patterns among numerical features and customer churn.

## 1. Import Libraries & Setup Visualization Style

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization styling
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["font.sans-serif"] = "Arial"
plt.rcParams["axes.edgecolor"] = "#cccccc"
plt.rcParams["axes.linewidth"] = 0.8

fig_dir = Path("../outputs/figures")
fig_dir.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

## 2. Load Processed Clean Dataset

In [ ]:
data_path = Path("../data/processed/telco_churn_clean.csv")
df = pd.read_csv(data_path)
print(f"Loaded clean dataset: {df.shape[0]:,} rows, {df.shape[1]} columns.")

## 3. Helper Function for Churn Rate Calculation & Visualization

In [ ]:
def calculate_and_plot_churn_rate(df, col_name, title, filename, xlabel, rotate=0):
    """
    Calculates group-level churn rate and generates a formatted bar chart.
    Churn Rate (%) = (Churned Customers in Category / Total Customers in Category) * 100
    """
    summary = df.groupby(col_name, observed=False)["Churn Value"].agg(
        Total="count",
        Churned="sum",
        Retained=lambda x: (x == 0).sum(),
        Churn_Rate=lambda x: (x.sum() / len(x)) * 100
    ).reset_index()
    summary["Churn_Rate"] = summary["Churn_Rate"].round(2)
    summary["Proportion_of_Total_Customers (%)"] = (summary["Total"] / len(df) * 100).round(2)
    
    display(summary)
    
    # Plot
    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(summary[col_name].astype(str), summary["Churn_Rate"], color="#3498db", edgecolor="#2980b9", width=0.55)
    ax.set_title(title, fontsize=14, pad=12, weight="bold")
    ax.set_ylabel("Churn Rate (%)", fontsize=12)
    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylim(0, max(summary["Churn_Rate"]) * 1.25)
    if rotate:
        plt.xticks(rotation=rotate, ha="right")
        
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f"{height:.1f}%",
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 4), textcoords="offset points",
                    ha="center", va="bottom", fontsize=11, weight="bold")
    
    plt.tight_layout()
    plt.savefig(fig_dir / filename, dpi=300)
    plt.show()
    return summary

## 4. Overall Churn Distribution

In [ ]:
overall_counts = df["Churn Value"].value_counts()
overall_pcts = (df["Churn Value"].value_counts(normalize=True) * 100).round(2)

overall_df = pd.DataFrame({
    "Status": ["Retained (0)", "Churned (1)"],
    "Customer Count": overall_counts.values,
    "Percentage (%)": overall_pcts.values
})
display(overall_df)

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(["Retained (0)", "Churned (1)"], overall_counts.values, color=["#2ecc71", "#e74c3c"], width=0.5, edgecolor="#333333")
ax.set_title("Overall Customer Churn Distribution (N = 7,043)", fontsize=14, pad=12, weight="bold")
ax.set_ylabel("Number of Customers", fontsize=12)
ax.set_ylim(0, 6000)
for bar, count in zip(bars, overall_counts.values):
    pct = (count / len(df)) * 100
    ax.annotate(f"{count:,} ({pct:.1f}%)",
                xy=(bar.get_x() + bar.get_width() / 2, count),
                xytext=(0, 4), textcoords="offset points",
                ha="center", va="bottom", fontsize=11, weight="bold")
plt.tight_layout()
plt.savefig(fig_dir / "01_overall_churn_distribution.png", dpi=300)
plt.show()

**Observation:** Out of 7,043 customers, 5,174 (73.46%) are retained and 1,869 (26.54%) churned, representing an approximate 3:1 baseline class ratio.

## 5. Churn by Contract Type

In [ ]:
contract_summary = calculate_and_plot_churn_rate(
    df, "Contract", "Churn Rate by Contract Type", "02_churn_by_contract.png", "Contract Type"
)

**Observation:** Customers on Month-to-month contracts have a churn rate of 42.71% (1,655 churned out of 3,875), compared to 11.27% for One year contracts and 2.83% for Two year contracts.

## 6. Churn by Tenure Months & Cohort Grouping

In [ ]:
bins = [0, 12, 24, 36, 48, 60, 72]
labels = ["0-12m", "13-24m", "25-36m", "37-48m", "49-60m", "61-72m"]
df["Tenure_Cohort"] = pd.cut(df["Tenure Months"], bins=bins, labels=labels, include_lowest=True)

tenure_summary = calculate_and_plot_churn_rate(
    df, "Tenure_Cohort", "Churn Rate across Tenure Cohorts", "03_churn_by_tenure_cohort.png", "Tenure Group (Months)"
)

**Observation:** Churn rate declines monotonically across tenure cohorts: 47.44% in the 0-12 month cohort, dropping to 28.71% (13-24m), 21.63% (25-36m), 19.03% (37-48m), 14.42% (49-60m), and 6.61% for customers with 61-72 months of tenure.

## 7. Churn by Internet Service Tier

In [ ]:
internet_summary = calculate_and_plot_churn_rate(
    df, "Internet Service", "Churn Rate by Internet Service Type", "04_churn_by_internet_service.png", "Internet Service Tier"
)

**Observation:** Fiber optic internet subscribers experience a churn rate of 41.89% (1,297 / 3,096), compared to 18.96% for DSL subscribers (459 / 2,421) and 7.40% for customers with no internet service (113 / 1,526).

## 8. Churn by Monthly Charges ($)

In [ ]:
monthly_stats = df.groupby("Churn Value")["Monthly Charges"].describe().round(2)
monthly_stats["IQR"] = (monthly_stats["75%"] - monthly_stats["25%"]).round(2)
display(monthly_stats)

fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(x="Churn Value", y="Monthly Charges", data=df, palette=["#2ecc71", "#e74c3c"], hue="Churn Value", legend=False, width=0.4, ax=ax)
ax.set_title("Monthly Charges Distribution by Churn Status", fontsize=14, pad=12, weight="bold")
ax.set_xticks([0, 1])
ax.set_xticklabels(["Retained (0)", "Churned (1)"])
ax.set_ylabel("Monthly Charges ($)", fontsize=12)
ax.set_xlabel("Customer Status", fontsize=12)
plt.tight_layout()
plt.savefig(fig_dir / "05_churn_by_monthly_charges.png", dpi=300)
plt.show()

**Observation:** Churned customers show higher monthly charges (mean: $74.44, median: $79.65) compared to retained customers (mean: $61.27, median: $64.43), reflecting a median difference of $15.22/month.

## 9. Churn by Payment Method

In [ ]:
payment_summary = calculate_and_plot_churn_rate(
    df, "Payment Method", "Churn Rate by Payment Method", "06_churn_by_payment_method.png", "Payment Method", rotate=15
)

**Observation:** Electronic check users exhibit the highest churn rate at 45.29% (1,071 / 2,365), whereas mailed check (19.11%), bank transfer (16.71%), and credit card (15.24%) have lower churn rates.

## 10. Churn by Tech Support Subscription

In [ ]:
tech_support_summary = calculate_and_plot_churn_rate(
    df, "Tech Support", "Churn Rate by Tech Support Subscription", "07_churn_by_tech_support.png", "Tech Support Status"
)

**Observation:** Customers without Tech Support have a churn rate of 41.64% (1,446 / 3,473), compared to 15.17% (310 / 2,044) for those with Tech Support.

## 11. Churn by Online Security Subscription

In [ ]:
online_security_summary = calculate_and_plot_churn_rate(
    df, "Online Security", "Churn Rate by Online Security Subscription", "08_churn_by_online_security.png", "Online Security Status"
)

**Observation:** Customers without Online Security have a churn rate of 41.77% (1,461 / 3,498), compared to 14.61% (295 / 2,019) for those with Online Security.

## 12. Churn by Senior Citizen Status

In [ ]:
senior_summary = calculate_and_plot_churn_rate(
    df, "Senior Citizen", "Churn Rate by Senior Citizen Status", "09_churn_by_senior_citizen.png", "Senior Citizen Status"
)

**Observation:** Senior citizens have a churn rate of 41.68% (476 / 1,142), compared to 23.61% (1,393 / 5,901) for non-senior customers.

## 13. Churn by Paperless Billing

In [ ]:
paperless_summary = calculate_and_plot_churn_rate(
    df, "Paperless Billing", "Churn Rate by Paperless Billing Enrollment", "10_churn_by_paperless_billing.png", "Paperless Billing"
)

**Observation:** Customers enrolled in paperless billing have a churn rate of 33.57% (1,400 / 4,171), compared to 16.33% (469 / 2,872) for those receiving paper invoices.

## 14. Numerical Relationships & Correlation Analysis

In [ ]:
num_cols = ["Tenure Months", "Monthly Charges", "Total Charges", "Churn Value"]
corr_matrix = df[num_cols].corr()
display(corr_matrix.round(3))

# Heatmap
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr_matrix, annot=True, fmt=".3f", cmap="Blues", vmin=-1, vmax=1, square=True, ax=ax, cbar_kws={"shrink": 0.8})
ax.set_title("Correlation Matrix: Numerical Features & Churn", fontsize=14, pad=12, weight="bold")
plt.tight_layout()
plt.savefig(fig_dir / "11_correlation_matrix.png", dpi=300)
plt.show()

# Scatter plot
fig, ax = plt.subplots(figsize=(9, 5.5))
sns.scatterplot(data=df, x="Tenure Months", y="Monthly Charges", hue="Churn Value", palette=["#2ecc71", "#e74c3c"], alpha=0.6, s=25, ax=ax)
ax.set_title("Tenure Months vs. Monthly Charges by Churn Outcome", fontsize=14, pad=12, weight="bold")
ax.set_xlabel("Tenure (Months)", fontsize=12)
ax.set_ylabel("Monthly Charges ($)", fontsize=12)
handles, _ = ax.get_legend_handles_labels()
ax.legend(handles=handles, labels=["Retained (0)", "Churned (1)"], title="Status", loc="upper right")
plt.tight_layout()
plt.savefig(fig_dir / "12_tenure_vs_monthly_charges_scatter.png", dpi=300)
plt.show()

**Observation:**
- `Tenure Months` and `Churn Value` have a negative linear correlation ($r = -0.352$), showing lower churn with longer tenure.
- `Monthly Charges` and `Churn Value` have a positive correlation ($r = +0.193$), indicating higher churn frequency with higher monthly charges.
- `Tenure Months` and `Total Charges` exhibit strong collinearity ($r = 0.826$), as total charges accumulate with customer tenure.